# V11 Settlement Enriched — Training Only

This notebook follows the original V11 Settlement training workflow. It does **not** fetch forecasts, fetch observations, perform parity checks, derive enrichment features, or run coverage gates. It only consumes feature files already materialized by the separate enrichment process.

Run `python scripts/run_v11_settlement_enrichment_pipeline.py` outside this notebook before starting training.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
CACHE = ROOT / 'data/calibration/v11_settlement_enriched_v1'
PREPARED = CACHE / 'prepared'
EXPERIMENT = CACHE / 'experiment'
STATIONS = ('KATL', 'KDAL')

## 1. Verify prepared training data

This is a read-only preflight. Training stops if the separate enrichment process has not produced all station/variant datasets.

In [ ]:
manifest_path = PREPARED / 'prepared_manifest.csv'
if not manifest_path.exists():
    raise FileNotFoundError(
        'Prepared enrichment data is missing. Run: '
        'python scripts/run_v11_settlement_enrichment_pipeline.py'
    )
manifest = pd.read_csv(manifest_path)
display(manifest)
expected = {'original_v11', 'cleaned_v11', 'observation_only', 'forecast_only', 'combined'}
assert expected.issubset(set(manifest['variant'])), 'One or more prepared variants are missing'
assert set(STATIONS).issubset(set(manifest['station_id'])), 'KATL or KDAL is missing'

## 2. Training contract

Architecture: XGBoost, LightGBM, CatBoost, and ridge stack. Target: settlement-first remaining warmup. Validation folds: 2021→2022, 2021–2022→2023, 2021–2023→2024, and 2021–2024→2025 with equal weights. Candidate selection never scores 2026.

In [ ]:
FOLDS = pd.DataFrame([
    {'train': '2021', 'validation': 2022},
    {'train': '2021-2022', 'validation': 2023},
    {'train': '2021-2023', 'validation': 2024},
    {'train': '2021-2024', 'validation': 2025},
])
display(FOLDS)

## 3. Frozen-hyperparameter screen

Train and compare cleaned V11, observation-only, forecast-only, and combined variants on the four validation folds. The original V11 base-model hyperparameters remain frozen during this stage.

In [ ]:
subprocess.run([
    sys.executable,
    str(ROOT / 'scripts/run_v11_settlement_enriched_experiment.py'),
    '--stage', 'screen',
    '--prepared-dir', str(PREPARED),
], cwd=ROOT, check=True)
display(pd.read_csv(EXPERIMENT / 'screen_candidate_ranking.csv'))

In [ ]:
winner = (EXPERIMENT / 'screen_winner.txt').read_text(encoding='utf-8').strip()
print('Selected enriched candidate:', winner)
screen_metrics = pd.read_csv(EXPERIMENT / 'screen_metrics.csv')
display(screen_metrics.query("scope == 'pooled'"))

## 4. Full tuning and final 2026 test

Fully tune only the selected enriched candidate with 30 Optuna trials and the existing CatBoost caps/stack search. The untouched 2026 test is evaluated once after candidate selection. Existing original-V11 artifacts are then loaded for comparison.

In [ ]:
subprocess.run([
    sys.executable,
    str(ROOT / 'scripts/run_v11_settlement_enriched_experiment.py'),
    '--stage', 'full',
    '--prepared-dir', str(PREPARED),
    '--winner', winner,
    '--optuna-trials', '30',
    '--stack-optuna-trials', '30',
], cwd=ROOT, check=True)

## 5. Results

In [ ]:
display(pd.read_csv(EXPERIMENT / 'full_validation_metrics.csv'))
display(pd.read_csv(EXPERIMENT / 'final_2026_metrics.csv'))
display(pd.read_csv(EXPERIMENT / 'promotion_decision.csv'))

In [ ]:
from IPython.display import Markdown, display
display(Markdown((EXPERIMENT / 'original_vs_enriched_report.md').read_text(encoding='utf-8')))

## 6. Diagnostics

In [ ]:
display(pd.read_csv(PREPARED / 'gated_feature_inventory.csv'))
display(pd.read_csv(PREPARED / 'feature_missingness_by_station_year.csv').head(50))
for station in STATIONS:
    importance = EXPERIMENT / 'full' / winner / f'{station}_feature_importance.csv'
    if importance.exists():
        display(pd.read_csv(importance).head(30))